In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision.transforms import v2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


# Define transformations for the training and validation sets


transform_train = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop((224, 224)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.CenterCrop((224, 224)),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# load oxford iiit pets dataset
train_dataset = datasets.OxfordIIITPet(root = './data',split = 'trainval',transform = transform_train, download = True)
val_dataset = datasets.OxfordIIITPet(root = './data', split = 'test', transform = transform_val, download = True)

train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 128, shuffle = False)


In [6]:
model = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)
for params in model.parameters():
    params.requires_grad = False

input_features = model.fc.in_features
model.fc = nn.Linear(input_features,37)
model = model.to(device)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [ ]:
epochs = 10 
criterion  = nn.CrossEntropyLoss()
optimizer = optim.Adam([{"params": model.fc.parameters(), "lr": 0.01},
                        {"params" : model.layer4.parameters(), "lr":0.001},
                        {"params" : model.layer3.parameters(), "lr": 0.0001},
                        {"params": model.layer2.parameters(), "lr": 0.0001}])
# here we have added different lr for different layers of the model. The final layer has the highest lr because it is randomly initialized and needs to learn the most, while the initial layers have the lowest lr because they are pre-trained and only need to be fine-tuned slightly.
step_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4,gamma= 0.1)
plateau_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 2)


In [8]:
training_loss = []
for epoch in range(epochs):
    model.train()
    trn_loss = 0
    i = 0
    if epoch == 2:
        for param in model.layer4.parameters():
            params.required_grad = True
        print("Unfroze layer 4")

    if epoch == 5:
        for params in model.layer3.parameters():
            params.requires_grad = True
        print("Unfroze layer 3")
    
    if epoch == 8:
        for params in model.layer2.parameters():
            params.requires_grad = True
        print("Unfroze layer 2")
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        trn_loss += loss.item()
        i += 1
        print(f"Batch {i}, Loss: {loss.item():.4f}")
        
    average_epoch_loss = trn_loss / len(train_loader)
    training_loss.append(average_epoch_loss)
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {average_epoch_loss:.10f}")
    step_scheduler.step()
    plateau_scheduler.step(average_epoch_loss)

Batch 1, Loss: 3.6511
Batch 2, Loss: 3.5914
Batch 3, Loss: 2.8430
Batch 4, Loss: 2.5712
Batch 5, Loss: 2.3588
Batch 6, Loss: 2.0675
Batch 7, Loss: 1.6376
Batch 8, Loss: 1.7621
Batch 9, Loss: 1.4677
Batch 10, Loss: 1.3598
Batch 11, Loss: 1.1369
Batch 12, Loss: 1.1611
Batch 13, Loss: 1.1588
Batch 14, Loss: 0.9399
Batch 15, Loss: 1.0097
Batch 16, Loss: 0.8623
Batch 17, Loss: 0.7234
Batch 18, Loss: 1.0541
Batch 19, Loss: 0.8754
Batch 20, Loss: 0.7599
Batch 21, Loss: 0.9629
Batch 22, Loss: 0.9901
Batch 23, Loss: 0.9299
Batch 24, Loss: 0.6952
Batch 25, Loss: 0.6589
Batch 26, Loss: 0.8977
Batch 27, Loss: 0.8488
Batch 28, Loss: 0.6622
Batch 29, Loss: 0.6034
Epoch 1/10, Training Loss: 1.3876162665
Batch 1, Loss: 0.5350
Batch 2, Loss: 0.4905
Batch 3, Loss: 0.7367
Batch 4, Loss: 0.4825
Batch 5, Loss: 0.6479
Batch 6, Loss: 0.4954
Batch 7, Loss: 0.5390
Batch 8, Loss: 0.6074
Batch 9, Loss: 0.5393
Batch 10, Loss: 0.5751
Batch 11, Loss: 0.5851
Batch 12, Loss: 0.5407
Batch 13, Loss: 0.4796
Batch 14, Lo

In [9]:
correct = 0; 
test_len = len(val_dataset)
model.eval()
with torch.no_grad():
    for batch_X,batch_y in val_loader:
        batch_correct_preds = 0
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        pred_labels = torch.argmax(y_pred,dim =1)
        batch_correct_preds += (pred_labels== batch_y).sum().item()
        correct += batch_correct_preds


print(correct)
accuracy = correct/test_len
print(f"Validation Accuracy: {accuracy*100:.10f}%")

3341
Validation Accuracy: 91.0602343963%
